# Entraînement & Sauvegarde — Modèle de Diffusion + Juges
## Pipeline final pour le Text-to-IMU (3 labels : rest, walk, run_jog)

Ce notebook reprend l'architecture du pipeline **VAE vs Diffusion (3 axes)**,
mais :
- **exclut le label `stairs`** (4) avant tout entraînement — la partie prompting
  ne gère que `rest / walk / run_jog`
- n'entraîne **que le modèle de diffusion** (pas le VAE, pas de comparaison)
- **sauvegarde tout** : modèle de diffusion, juges, et statistiques de
  normalisation — pour pouvoir ensuite générer des données à partir d'un
  simple prompt en quelques lignes, sans tout réentraîner.


## 1. Configuration & Imports

In [1]:
import gc
import sys
import json
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pyarrow.parquet as pq

from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("===== DEBUG PYTHON / CUDA =====")
print("Python executable:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. This script requires a GPU.")

device = torch.device("cuda:0")
print("Device utilisé :", device)
print("GPU utilisée :", torch.cuda.get_device_name(0))
print("================================")

# ── Chemin du fichier parquet (dataset complet, non filtré sur les labels) ──
path = r"C:\Users\adril\Downloads\unified_dataset_filtered_4_full.parquet"

# ── Labels gérés par la partie prompting (stairs exclu) ──────────────────────
ALLOWED_LABELS = {0, 2, 3}   # rest_inactive, walk, run_jog — PAS stairs (4)

# ── Hyper-paramètres fenêtrage ────────────────────────────────────────────────
WINDOW_SIZE   = 50
STRIDE        = 25
BATCH_SIZE    = 4096
FS            = 25

# ── Diffusion ─────────────────────────────────────────────────────────────────
DIFF_EPOCHS   = 50
N_STEPS       = 200
TIME_EMB_DIM  = 32
LABEL_EMB_DIM = 16

# ── Juges ──────────────────────────────────────────────────────────────────────
JUDGE_EPOCHS  = 30

# ── Sortie ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR = "saved_models"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration OK.")


===== DEBUG PYTHON / CUDA =====
Python executable: D:\IMDS\MALMO\.venv\Scripts\python.exe
Torch version: 2.6.0+cu124
CUDA available: True
Device utilisé : cuda:0
GPU utilisée : NVIDIA GeForce RTX 4060 Laptop GPU
Configuration OK.


## 2. Chargement des données — 3 axes, labels filtrés
On exclut `stairs` (label 4) **dès le chargement**, avant le mapping des
labels et avant le split. Ainsi `n_classes = 3` partout dans la suite,
exactement comme attendu par le classifieur de prompt (`rest / walk / run_jog`).


In [2]:
X_3ax  = []
labels = []
groups = []

parquet_file = pq.ParquetFile(path)
print(f"Nombre de paquets à traiter : {parquet_file.num_row_groups}")

for i in range(parquet_file.num_row_groups):
    chunk = parquet_file.read_row_group(
        i,
        columns=[
            "acc_x", "acc_y", "acc_z",
            "global_activity_id", "dataset", "subject_id", "session_id",
        ],
    ).to_pandas()

    # ── Filtrage des labels non désirés (stairs) AVANT tout traitement ───────
    chunk = chunk[chunk["global_activity_id"].isin(ALLOWED_LABELS)]
    if len(chunk) == 0:
        del chunk
        gc.collect()
        continue

    for gk, gdf in chunk.groupby(["dataset", "subject_id", "session_id"]):
        ax_sig = gdf[["acc_x", "acc_y", "acc_z"]].values.astype(np.float32)
        acts   = gdf["global_activity_id"].values

        if len(ax_sig) < WINDOW_SIZE:
            continue

        for j in range(0, len(ax_sig) - WINDOW_SIZE + 1, STRIDE):
            X_3ax.append(ax_sig[j: j + WINDOW_SIZE])
            lbl = Counter(acts[j: j + WINDOW_SIZE]).most_common(1)[0][0]
            labels.append(lbl)
            groups.append(str(gk))

    del chunk
    gc.collect()

    if (i + 1) % 5 == 0:
        print(f"Paquet {i + 1}/{parquet_file.num_row_groups} terminé...")

X_3ax  = np.asarray(X_3ax,  dtype=np.float32)   # (N, 50, 3)
labels = np.asarray(labels)
groups = np.asarray(groups)

print(f"\nChargement terminé. Fenêtres : {len(X_3ax):,}")
print("Labels présents :", sorted(np.unique(labels)))
assert 4 not in np.unique(labels), "Le label stairs (4) n'a pas été exclu correctement !"


Nombre de paquets à traiter : 25
Paquet 5/25 terminé...
Paquet 10/25 terminé...
Paquet 15/25 terminé...
Paquet 20/25 terminé...

Chargement terminé. Fenêtres : 460,792
Labels présents : [np.int32(0), np.int32(2), np.int32(3)]


## 3. Mapping des labels & split train/val/test

In [3]:
ACTIVITY_NAMES = {0: "rest_inactive", 2: "walk", 3: "run_jog"}

unique_labels     = sorted(np.unique(labels))
label_mapping     = {int(old): int(new) for new, old in enumerate(unique_labels)}
inv_label_mapping = {int(new): int(old) for old, new in label_mapping.items()}
labels_mapped     = np.asarray([label_mapping[l] for l in labels], dtype=np.int64)
n_classes         = len(unique_labels)

print("Mapping des labels :")
for old, new in label_mapping.items():
    print(f"  {old} ({ACTIVITY_NAMES.get(old)}) → {new}")
print(f"Nombre de classes : {n_classes}")

# ── Split par session (stratifié groupe) ──────────────────────────────────────
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=SEED)
train_val_idx, test_idx = next(gss_test.split(X_3ax, labels_mapped, groups=groups))

X_tv = X_3ax[train_val_idx];  y_tv = labels_mapped[train_val_idx];  g_tv = groups[train_val_idx]
X_3ax_test = X_3ax[test_idx]; y_test = labels_mapped[test_idx]

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.2222, random_state=SEED)
tr_idx, va_idx = next(gss_val.split(X_tv, y_tv, groups=g_tv))

X_3ax_train = X_tv[tr_idx];  y_train = y_tv[tr_idx]
X_3ax_val   = X_tv[va_idx];  y_val   = y_tv[va_idx]

print(f"Train : {len(X_3ax_train):,}   Val : {len(X_3ax_val):,}   Test : {len(X_3ax_test):,}")


Mapping des labels :
  0 (rest_inactive) → 0
  2 (walk) → 1
  3 (run_jog) → 2
Nombre de classes : 3
Train : 336,975   Val : 78,514   Test : 45,303


## 4. Normalisation (z-score, stats calculées sur train uniquement)

In [4]:
ax_mean = X_3ax_train.mean(axis=(0, 1), keepdims=True)   # (1, 1, 3)
ax_std  = X_3ax_train.std(axis=(0, 1), keepdims=True)

def norm_3ax(x):
    return (x - ax_mean) / (ax_std + 1e-8)

def denorm_3ax(x):
    return x * (ax_std + 1e-8) + ax_mean

X_3ax_train_n = norm_3ax(X_3ax_train)
X_3ax_val_n   = norm_3ax(X_3ax_val)
X_3ax_test_n  = norm_3ax(X_3ax_test)

print("ax_mean :", ax_mean.flatten())
print("ax_std  :", ax_std.flatten())

# ── Tenseurs PyTorch (N, 3, L) ────────────────────────────────────────────────
def to_t(a, dtype=torch.float32):
    return torch.tensor(a, dtype=dtype)

Xtn_t = to_t(X_3ax_train_n).permute(0, 2, 1)
Xvn_t = to_t(X_3ax_val_n).permute(0, 2, 1)
Xen_t = to_t(X_3ax_test_n).permute(0, 2, 1)

ytr_t = to_t(y_train, torch.long)
yva_t = to_t(y_val,   torch.long)
yte_t = to_t(y_test,  torch.long)

train_ds = TensorDataset(Xtn_t, ytr_t)
val_ds   = TensorDataset(Xvn_t, yva_t)
test_ds  = TensorDataset(Xen_t, yte_t)

def make_loader(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=True)

train_loader = make_loader(train_ds, shuffle=True)
val_loader   = make_loader(val_ds)
test_loader  = make_loader(test_ds)

print("Tenseurs prêts. Shape batch exemple :", next(iter(train_loader))[0].shape)


ax_mean : [-0.03272235  0.01561515 -0.01650173]
ax_std  : [0.9859714  1.0075629  0.97911924]
Tenseurs prêts. Shape batch exemple : torch.Size([4096, 3, 50])


## 5. Architecture — Modèle de Diffusion conditionnel (DDPM)

In [5]:
class ConditionalDiffusionNet(nn.Module):
    def __init__(self, n_classes, signal_len=50,
                 time_emb_dim=32, label_emb_dim=16):
        super().__init__()
        self.label_emb  = nn.Embedding(n_classes, label_emb_dim)
        self.time_mlp   = nn.Sequential(
            nn.Linear(1, time_emb_dim), nn.ReLU(),
            nn.Linear(time_emb_dim, time_emb_dim), nn.ReLU(),
        )
        self.conv1 = nn.Conv1d(3, 64, 5, padding=2)
        self.conv2 = nn.Conv1d(64, 64, 5, padding=2)
        self.conv3 = nn.Conv1d(64, 64, 5, padding=2)
        self.conv4 = nn.Conv1d(64, 3, 5, padding=2)
        self.cond_project = nn.Linear(time_emb_dim + label_emb_dim, 64)
        self.relu = nn.ReLU()

    def forward(self, x, t_float, y):
        cond = torch.cat([self.time_mlp(t_float), self.label_emb(y)], 1)
        cp   = self.cond_project(cond).unsqueeze(-1)
        h    = self.relu(self.conv1(x)) + cp
        h    = self.relu(self.conv2(h)) + cp
        h    = self.relu(self.conv3(h))
        return self.conv4(h)


class DiffusionContext:
    def __init__(self, n_steps=200, signal_len=50, device="cpu"):
        self.n_steps  = n_steps
        self.device   = device
        self.beta     = torch.linspace(1e-4, 0.02, n_steps, device=device)
        self.alpha    = 1.0 - self.beta
        self.alpha_cp = torch.cumprod(self.alpha, 0)

    def add_noise(self, x0, t):
        noise  = torch.randn_like(x0)
        t_flat = t.view(-1).long()
        sa     = torch.sqrt(self.alpha_cp[t_flat]).view(-1, 1, 1)
        sm     = torch.sqrt(1 - self.alpha_cp[t_flat]).view(-1, 1, 1)
        return sa * x0 + sm * noise, noise

    @torch.no_grad()
    def sample(self, model, n_samples, class_id, signal_len):
        model.eval()
        x = torch.randn(n_samples, 3, signal_len, device=self.device)
        y = torch.full((n_samples,), class_id, dtype=torch.long, device=self.device)

        for i in reversed(range(self.n_steps)):
            tf  = torch.full((n_samples, 1), i, dtype=torch.float32, device=self.device)
            pn  = model(x, tf, y)
            bt  = self.beta[i]
            sat = torch.sqrt(self.alpha[i])
            smt = torch.sqrt(1 - self.alpha_cp[i])
            mu  = (1 / sat) * (x - (bt / smt) * pn)
            x   = mu + (torch.sqrt(bt) * torch.randn_like(x) if i > 0 else 0)
        return x   # (n, 3, signal_len)

print("Architecture Diffusion définie (3 canaux : X, Y, Z).")


Architecture Diffusion définie (3 canaux : X, Y, Z).


> **Note technique :** dans le notebook de comparaison original, la dernière
> couche du réseau de débruitage produisait 1 canal (`Conv1d(64, 1, ...)`),
> ce qui est incohérent avec une entrée/sortie à 3 axes. Ici `conv4` produit
> bien **3 canaux** (`Conv1d(64, 3, ...)`), pour que le bruit prédit ait la
> même forme que le signal bruité `x` — condition nécessaire au DDPM.


## 6. Architectures des juges (3 axes en entrée)

In [6]:
class DeepConvLSTM(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(3, 64, 5, padding=2), nn.ReLU(),
            nn.Conv1d(64, 64, 5, padding=2), nn.ReLU(),
        )
        self.lstm = nn.LSTM(64, 128, num_layers=2, batch_first=True)
        self.fc   = nn.Sequential(
            nn.Linear(128, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )
    def forward(self, x):
        h, _ = self.lstm(self.conv(x).transpose(1, 2))
        return self.fc(h[:, -1, :])


class CNNSimple(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(3, 64, 3, padding=1), nn.ReLU(),
            nn.Conv1d(64, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, n_classes),
        )
    def forward(self, x):
        return self.net(x)


class MLPSimple(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(WINDOW_SIZE * 3, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, n_classes),
        )
    def forward(self, x):
        return self.net(x)

print("Architectures juges définies (entrée 3 canaux).")


Architectures juges définies (entrée 3 canaux).


## 7. Fonctions d'entraînement

In [7]:
def train_diffusion(model, ctx, loader, val_loader=None, epochs=DIFF_EPOCHS):
    opt  = optim.Adam(model.parameters(), lr=1e-4)
    crit = nn.MSELoss()
    for ep in range(epochs):
        model.train()
        tot_loss = tot_n = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            t       = torch.randint(0, ctx.n_steps, (xb.size(0),), device=device)
            xt, eps = ctx.add_noise(xb, t)
            tf      = t.float().unsqueeze(-1)
            pred    = model(xt, tf, yb)
            loss    = crit(pred, eps)
            if not torch.isfinite(loss):
                return
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot_loss += loss.item() * xb.size(0)
            tot_n    += xb.size(0)
        tl = tot_loss / tot_n
        vl_str = ""
        if val_loader is not None:
            model.eval(); vtot = vn = 0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    t = torch.randint(0, ctx.n_steps, (xb.size(0),), device=device)
                    xt, eps = ctx.add_noise(xb, t)
                    tf = t.float().unsqueeze(-1)
                    l  = crit(model(xt, tf, yb), eps)
                    vtot += l.item() * xb.size(0); vn += xb.size(0)
            vl_str = f" | ValLoss {vtot/vn:.6f}"
        print(f"Diff Ep {ep+1:03d} | TrainLoss {tl:.6f}{vl_str}")


def train_judge(model, train_loader, val_loader=None, epochs=JUDGE_EPOCHS):
    opt  = optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    for ep in range(epochs):
        model.train()
        tot_loss = tot_corr = tot_n = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss   = crit(logits, yb)
            if not torch.isfinite(loss):
                return
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot_loss += loss.item() * xb.size(0)
            tot_corr += (logits.argmax(1) == yb).sum().item()
            tot_n    += xb.size(0)
        tl = tot_loss / tot_n; ta = tot_corr / tot_n
        if val_loader is not None:
            vl, va = eval_judge(model, val_loader)
            print(f"[{model.__class__.__name__}] Ep {ep+1:03d} "
                  f"| TrLoss {tl:.4f} TrAcc {ta:.4f} "
                  f"| VaLoss {vl:.4f} VaAcc {va:.4f}")
        else:
            print(f"[{model.__class__.__name__}] Ep {ep+1:03d} "
                  f"| TrLoss {tl:.4f} TrAcc {ta:.4f}")


def eval_judge(model, loader):
    model.eval()
    crit = nn.CrossEntropyLoss()
    tot_loss = tot_corr = tot_n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss   = crit(logits, yb)
            tot_loss += loss.item() * xb.size(0)
            tot_corr += (logits.argmax(1) == yb).sum().item()
            tot_n    += xb.size(0)
    return tot_loss / tot_n, tot_corr / tot_n

print("Fonctions d'entraînement définies.")


Fonctions d'entraînement définies.


## 8. Entraînement principal

In [8]:
# ── Diffusion ──────────────────────────────────────────────────────────────────
diff_model   = ConditionalDiffusionNet(n_classes, WINDOW_SIZE, TIME_EMB_DIM, LABEL_EMB_DIM).to(device)
diff_context = DiffusionContext(N_STEPS, WINDOW_SIZE, device)

print("🟠 Entraînement du modèle de diffusion conditionnel")
train_diffusion(diff_model, diff_context, train_loader, val_loader, epochs=DIFF_EPOCHS)

# ── Juges ──────────────────────────────────────────────────────────────────────
judges = [
    DeepConvLSTM(n_classes).to(device),
    CNNSimple(n_classes).to(device),
    MLPSimple(n_classes).to(device),
]

for judge in judges:
    print(f"\n🟢 Entraînement Juge : {judge.__class__.__name__}")
    train_judge(judge, train_loader, val_loader, epochs=JUDGE_EPOCHS)
    judge.eval()

diff_model.eval()
print("\n✅ Entraînement terminé (Diffusion + 3 Juges).")


🟠 Entraînement du modèle de diffusion conditionnel
Diff Ep 001 | TrainLoss 1.148872 | ValLoss 0.968371
Diff Ep 002 | TrainLoss 0.940424 | ValLoss 0.901725
Diff Ep 003 | TrainLoss 0.848524 | ValLoss 0.786574
Diff Ep 004 | TrainLoss 0.712860 | ValLoss 0.654103
Diff Ep 005 | TrainLoss 0.588996 | ValLoss 0.519920
Diff Ep 006 | TrainLoss 0.471263 | ValLoss 0.442898
Diff Ep 007 | TrainLoss 0.411857 | ValLoss 0.398851
Diff Ep 008 | TrainLoss 0.384182 | ValLoss 0.379102
Diff Ep 009 | TrainLoss 0.363793 | ValLoss 0.375010
Diff Ep 010 | TrainLoss 0.351801 | ValLoss 0.363766
Diff Ep 011 | TrainLoss 0.345537 | ValLoss 0.354529
Diff Ep 012 | TrainLoss 0.335102 | ValLoss 0.351901
Diff Ep 013 | TrainLoss 0.325221 | ValLoss 0.336821
Diff Ep 014 | TrainLoss 0.315686 | ValLoss 0.330023
Diff Ep 015 | TrainLoss 0.305933 | ValLoss 0.318494
Diff Ep 016 | TrainLoss 0.298553 | ValLoss 0.312972
Diff Ep 017 | TrainLoss 0.292159 | ValLoss 0.307824
Diff Ep 018 | TrainLoss 0.284891 | ValLoss 0.299079
Diff Ep 019 |

## 9. Vérification rapide sur le test set (sanity check)

In [9]:
print("Accuracy des juges sur le test set réel (sanity check avant sauvegarde) :")
for judge in judges:
    _, test_acc = eval_judge(judge, test_loader)
    print(f"  {judge.__class__.__name__:<16} : {test_acc:.4f}")


Accuracy des juges sur le test set réel (sanity check avant sauvegarde) :
  DeepConvLSTM     : 0.9815
  CNNSimple        : 0.9848
  MLPSimple        : 0.9783


## 10. Sauvegarde des modèles
On sauvegarde :
- les poids du modèle de diffusion (`.pth`)
- les poids des 3 juges (`.pth`)
- les statistiques de normalisation + le mapping des labels (`.json`), pour
  pouvoir dénormaliser/remapper correctement à l'inférence sans avoir à
  recharger tout le dataset.


In [10]:
# ── Modèle de diffusion ────────────────────────────────────────────────────────
torch.save(diff_model.state_dict(), f"{OUTPUT_DIR}/diffusion_model.pth")

# ── Juges ──────────────────────────────────────────────────────────────────────
for judge in judges:
    torch.save(judge.state_dict(), f"{OUTPUT_DIR}/{judge.__class__.__name__}.pth")

# ── Métadonnées (normalisation + mapping + hyperparamètres) ──────────────────
metadata = {
    "window_size":    WINDOW_SIZE,
    "fs":             FS,
    "n_steps":        N_STEPS,
    "time_emb_dim":   TIME_EMB_DIM,
    "label_emb_dim":  LABEL_EMB_DIM,
    "n_classes":      n_classes,
    "label_mapping":      {str(k): int(v) for k, v in label_mapping.items()},
    "inv_label_mapping":  {str(k): int(v) for k, v in inv_label_mapping.items()},
    "activity_names":     {str(int(k)): v for k, v in ACTIVITY_NAMES.items()},
    "ax_mean": ax_mean.flatten().tolist(),   # [mean_x, mean_y, mean_z]
    "ax_std":  ax_std.flatten().tolist(),    # [std_x,  std_y,  std_z]
}

with open(f"{OUTPUT_DIR}/metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Modèles et métadonnées sauvegardés dans ./{OUTPUT_DIR}/")
print("Contenu :")
import os
for fname in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  - {fname}")


✅ Modèles et métadonnées sauvegardés dans ./saved_models/
Contenu :
  - CNNSimple.pth
  - DeepConvLSTM.pth
  - MLPSimple.pth
  - diffusion_model.pth
  - metadata.json
